In [ ]:
# Challenge 4: Programming an Agent Workflow (SequentialAgent)
import os
import re
import time
import warnings

from google.adk.agents.llm_agent import LlmAgent
from google.adk.agents.sequential_agent import SequentialAgent
from google.adk.runners import Runner
from google.adk.sessions.in_memory_session_service import (
    InMemorySessionService,
)
from google.adk.tools import google_search
from google.adk.tools.agent_tool import AgentTool
from google.genai import types
from IPython.display import clear_output, display as ipy_display
import ipywidgets as widgets
import requests
import vertexai

# REQUIRED FIX — disable LiteLLM MCP subsystem
os.environ["LITELLM_DISABLE_MCP"] = "true"

# Suppress experimental JSON schema warning
warnings.filterwarnings(
    "ignore",
    message=".*JSON_SCHEMA_FOR_FUNC_DECL is enabled.*"
)

# ------------------------------------------------------------
# 1. INITIALIZATION & SETUP
# ------------------------------------------------------------

PROJECT_ID = !gcloud config get-value project
PROJECT_ID = PROJECT_ID[0]
LOCATION = "us-central1"
MODEL_LOCATION = "global"
STAGING_BUCKET = "gs://agent-engine-staging-bucket-dar2"

# Define project information manually if the above code didn't work
if PROJECT_ID == "(unset)" or not PROJECT_ID:
    PROJECT_ID = "qwiklabs-gcp-01-6d489226baf6"

print(f"Project ID: {PROJECT_ID}")

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_QUOTA_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["ADK_SUPPRESS_GEMINI_LITELLM_WARNINGS"] = "true"

vertexai.init(
    project=PROJECT_ID,
    location=MODEL_LOCATION,
    staging_bucket=STAGING_BUCKET,
)

session_service = InMemorySessionService()

prompt_logs = []
response_logs = []

# ------------------------------------------------------------
# CALLBACKS FOR LOGGING & VALIDATION
# ------------------------------------------------------------


def validate_user_input(session, message):
    """Callback to validate user input before sending to model."""
    if not message or not message.parts:
        raise ValueError("Empty message encountered.")

    user_text = " ".join(
        [part.text for part in message.parts if part.text]
    ).strip()

    if len(user_text) < 2:
        raise ValueError(
            "Query is too short. Please provide a valid request."
        )

    malicious_patterns = [
        r"(?i)system\s+prompt",
        r"(?i)ignore\s+previous",
        r"(?i)delete\s+from",
        r"(?i)drop\s+table",
        r"(?i)select\s+\*\s+from",
    ]
    for pattern in malicious_patterns:
        if re.search(pattern, user_text):
            raise ValueError(
                "Security violation: Input contains prohibited commands "
                "or prompt injection."
            )

    non_us_indicators = [
        "london", "paris", "tokyo", "berlin", "beijing", "sydney", "rome",
        "canada", "mexico", "france", "germany", "uk", "united kingdom",
        "japan", "australia",
    ]
    normalized_text = user_text.lower()
    is_weather_query = any(
        w in normalized_text for w in ["weather", "forecast", "temperature"]
    )
    for indicator in non_us_indicators:
        if (
            f", {indicator}" in normalized_text
            or normalized_text == indicator
            or (is_weather_query and indicator in normalized_text)
        ):
            raise ValueError(
                f"NWS Forecast only supports United States locations. "
                f"'{user_text}' is recognized as outside the US."
            )

    return message


def log_user_prompt(session, message):
    """Callback to log user prompts."""
    user_text = " ".join(
        [part.text for part in message.parts if part.text]
    ).strip()
    log_entry = f"[Session: {session.session_id}] Prompt: {user_text}"
    prompt_logs.append(log_entry)
    print(f"[CALLBACK LOG - USER PROMPT]: {user_text}")


def log_model_response(session, message):
    """Callback to log model responses."""
    model_text = " ".join(
        [part.text for part in message.parts if part.text]
    ).strip()
    log_entry = f"[Session: {session.session_id}] Response: {model_text}"
    response_logs.append(log_entry)
    print(f"[CALLBACK LOG - MODEL RESPONSE]: {model_text}")


# ------------------------------------------------------------
# 2. DEFINE TOOLS FOR WEATHER AGENT
# ------------------------------------------------------------


def geocode_place_free(address: str) -> str:
    """Geocode an address into latitude and longitude coordinates."""
    base_url = "https://nominatim.openstreetmap.org/search"
    params = {"q": address, "format": "json", "limit": 1}
    headers = {
        "User-Agent": "RussellWeatherApp/1.0 (russell.f.zimmermann@saic.com)"
    }
    try:
        response = requests.get(
            base_url, params=params, headers=headers, timeout=10
        )
        response.raise_for_status()
        data = response.json()
        if data:
            return f"{data[0]['lat']}, {data[0]['lon']}"
    except Exception:
        pass
    return "Error: Location not found."


def get_nws_forecast(latitude: float, longitude: float) -> str:
    """Retrieve the National Weather Service (NWS) forecast."""
    headers = {
        "User-Agent": "RussellWeatherApp/1.0 (russell.f.zimmermann@saic.com)"
    }
    points_url = f"https://api.weather.gov/points/{latitude},{longitude}"
    try:
        points_response = requests.get(
            points_url, headers=headers, timeout=10
        )
        points_response.raise_for_status()
        forecast_url = points_response.json()["properties"]["forecast"]
        forecast_response = requests.get(
            forecast_url, headers=headers, timeout=10
        )
        forecast_response.raise_for_status()
        periods = forecast_response.json()["properties"]["periods"]
        formatted = []
        for period in periods[:2]:
            formatted.append(
                f"{period['name']}: {period['temperature']}°"
                f"{period['temperatureUnit']}, {period['shortForecast']}."
            )
        return "\n\n".join(formatted)
    except Exception as e:
        return f"Error: {e}"


# ------------------------------------------------------------
# 3. DEFINE CHALLENGE 4 WORKFLOW AGENTS
#    (Instructions 2 & 3: Greeter + Sequential Answer Team:
#     Search -> Critique -> Refine)
# ------------------------------------------------------------

greeter_instruction = (
    "You are a friendly Greeter Agent. Warmly welcome the user, introduce "
    "yourself, and explain that this system can provide US weather forecasts "
    "or answer complex questions using a 3-step Search, Critique, and Refine "
    "workflow team."
)

weather_instruction = (
    "You are a helpful weather assistant. "
    "When a user provides a location, use your geocoding and forecast tools "
    "to find the weather forecast for that location. "
    "IMPORTANT: Always explicitly state the identified state (or region) "
    "and country for the city you chose in your final response to the user."
)

# Isolated Google Search worker wrapped in AgentTool so google_search is never
# mixed with transfer_to_agent function declarations.
google_search_worker_instruction = (
    "Use the google_search tool to find accurate, up-to-date facts and data "
    "for the given query."
)

search_instruction = (
    "You are the Search Agent (Step 1 of the Answer Team). "
    "Use your `google_search_worker` tool to find accurate, up-to-date data "
    "to answer the user's question, and provide a comprehensive initial "
    "draft answer based on the search findings."
)

critique_instruction = (
    "You are the Critique Agent (Step 2 of the Answer Team). "
    "Review the initial response generated by the Search Agent: "
    "{initial_answer}\n\n"
    "Evaluate its accuracy, completeness, structure, and clarity. "
    "Provide 2-3 specific, constructive suggestions on how the response "
    "can be improved before returning it to the user."
)

refine_instruction = (
    "You are the Refine Agent (Step 3 of the Answer Team). "
    "Take the Search Agent's initial response:\n{initial_answer}\n\n"
    "And the Critique Agent's suggestions:\n{critique_feedback}\n\n"
    "Rewrite and polish the response so it directly, clearly, and "
    "accurately answers the user's question incorporating the suggested "
    "improvements. Return the final refined response for the user."
)

root_instruction = (
    "You are the Root Coordinating Agent. Route user requests to the "
    "appropriate sub-agent:\n"
    "- If the user is saying hello, greeting you, or asking what you can do, "
    "delegate to `greeter_agent` (or `greeter_agent_pro`).\n"
    "- If the user asks about US weather, temperature, or forecasts, "
    "delegate to `weather_agent` (or `weather_agent_pro`).\n"
    "- For all general knowledge, factual, research, or current events "
    "questions, delegate to the `answer_team` (or `answer_team_pro`) "
    "sequential workflow agent."
)


def build_hierarchy(model_name: str, suffix: str = ""):
    """Build the Challenge 4 multi-agent hierarchy for a given model."""
    greeter = LlmAgent(
        name=f"greeter_agent{suffix}",
        model=model_name,
        description="Greets the user and explains what the system can do.",
        instruction=greeter_instruction,
    )

    weather = LlmAgent(
        name=f"weather_agent{suffix}",
        model=model_name,
        description="Retrieves US weather forecasts given a city or location.",
        instruction=weather_instruction,
        tools=[geocode_place_free, get_nws_forecast],
    )

    # Isolated search worker wrapped with AgentTool to avoid 400 tool conflict
    search_worker = LlmAgent(
        name=f"google_search_worker{suffix}",
        model=model_name,
        description="Executes Google Search queries to retrieve factual data.",
        instruction=google_search_worker_instruction,
        tools=[google_search],
    )

    # 3.1 Search Agent
    search = LlmAgent(
        name=f"search_agent{suffix}",
        model=model_name,
        description="Finds data using Google Search to answer the question.",
        instruction=search_instruction,
        tools=[AgentTool(agent=search_worker)],
        output_key="initial_answer",
    )

    # 3.2 Critique Agent
    critique = LlmAgent(
        name=f"critique_agent{suffix}",
        model=model_name,
        description=(
            "Reviews the initial response and suggests improvements."
        ),
        instruction=critique_instruction,
        output_key="critique_feedback",
    )

    # 3.3 Refine Agent
    refine = LlmAgent(
        name=f"refine_agent{suffix}",
        model=model_name,
        description=(
            "Rewrites the response based on Critique Agent's suggestions."
        ),
        instruction=refine_instruction,
        output_key="final_refined_answer",
    )

    # Instruction 3: SequentialAgent Answer Team
    answer_team = SequentialAgent(
        name=f"answer_team{suffix}",
        description=(
            "Sequential workflow team that searches, critiques, and refines "
            "answers to questions."
        ),
        sub_agents=[search, critique, refine],
    )

    root = LlmAgent(
        name=f"root_agent{suffix}",
        model=model_name,
        description=(
            "Root coordinating agent that delegates to greeter_agent, "
            "weather_agent, and answer_team."
        ),
        instruction=root_instruction,
        sub_agents=[greeter, weather, answer_team],
    )
    return root


# --- FLASH & PRO RUNNERS ---
root_agent = build_hierarchy("gemini-2.5-flash", suffix="")
agent_flash = root_agent
runner_flash = Runner(
    agent=agent_flash,
    app_name="Challenge4FlashWorkflowApp",
    session_service=session_service,
    auto_create_session=True,
)

agent_pro = build_hierarchy("gemini-2.5-pro", suffix="_pro")
runner_pro = Runner(
    agent=agent_pro,
    app_name="Challenge4ProWorkflowApp",
    session_service=session_service,
    auto_create_session=True,
)

runner_flash.pre_run_callbacks = [validate_user_input, log_user_prompt]
runner_flash.post_run_callbacks = [log_model_response]

runner_pro.pre_run_callbacks = [validate_user_input, log_user_prompt]
runner_pro.post_run_callbacks = [log_model_response]

# ------------------------------------------------------------
# 4. INTERACTIVE CHAT INTERFACE
# ------------------------------------------------------------

session_id = "challenge4_workflow_session"
user_id = "russell"

chat_output = widgets.Output()

model_dropdown = widgets.Dropdown(
    options=[("Gemini 2.5 Flash", "flash"), ("Gemini 2.5 Pro", "pro")],
    value="flash",
    description="Select Model",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="260px"),
)

custom_css = """
<style>
.widget-dropdown select {
    background-color: #212529 !important;
    color: #ffffff !important;
    border: 1px solid #495057 !important;
}
</style>
"""
css_widget = widgets.HTML(custom_css)

user_text_input = widgets.Text(
    placeholder=(
        "Say hello, ask for US weather, or ask a question for the "
        "Search->Critique->Refine team..."
    ),
    layout=widgets.Layout(width="60%"),
)

send_button = widgets.Button(
    description="Send",
    button_style="info",
    layout=widgets.Layout(width="15%"),
)
exit_button = widgets.Button(
    description="End Chat",
    button_style="danger",
    layout=widgets.Layout(width="15%"),
)

top_row = widgets.HBox(
    [model_dropdown],
    layout=widgets.Layout(grid_gap="20px"),
)
bottom_row = widgets.HBox(
    [user_text_input, send_button, exit_button],
    layout=widgets.Layout(grid_gap="10px"),
)
controls = widgets.VBox(
    [top_row, bottom_row],
    layout=widgets.Layout(margin="10px 0px"),
)

interface_box = widgets.VBox([chat_output, controls])


def append_message(sender, text, is_agent=False):
    """Append a formatted message to the chat output widget."""
    with chat_output:
        color = "#0d6efd" if not is_agent else "#198754"
        sender_name = "You" if not is_agent else sender
        ipy_display(widgets.HTML(
            f"<b style='color: {color};'>{sender_name}:</b> {text}<br>"
        ))


def disable_interface():
    """Disable interactive widgets when the conversation ends."""
    model_dropdown.disabled = True
    user_text_input.disabled = True
    send_button.disabled = True
    exit_button.disabled = True
    with chat_output:
        ipy_display(widgets.HTML(
            "<hr><b style='color: #dc3545;'>Conversation ended.</b>"
        ))


def handle_submit(sender):
    """Handle user query submission and stream workflow agent events."""
    user_query = user_text_input.value.strip()
    if not user_query:
        return
    user_text_input.value = ""

    append_message(user_id, user_query, is_agent=False)
    if user_query.lower() in ["exit", "quit"]:
        disable_interface()
        return

    selected_model_key = model_dropdown.value
    active_runner = runner_pro if selected_model_key == "pro" else runner_flash
    model_display_name = (
        "Gemini 2.5 Pro" if selected_model_key == "pro"
        else "Gemini 2.5 Flash"
    )

    with chat_output:
        ipy_display(widgets.HTML(
            f"<small style='color: #6c757d; font-style: italic;'>"
            f"[Routing message to Root Agent ({model_display_name})]</small>"
        ))

    message_content = types.Content(
        role="user",
        parts=[types.Part.from_text(text=user_query)],
    )
    try:
        events = active_runner.run(
            user_id=user_id,
            session_id=session_id,
            new_message=message_content,
        )
        for event in events:
            author = getattr(event, "author", "Agent")
            actions = getattr(event, "actions", None)
            transfer_target = getattr(actions, "transfer_to_agent", None)
            if transfer_target:
                with chat_output:
                    ipy_display(widgets.HTML(
                        f"<small style='color: #6c757d;'>"
                        f"&#8627; Delegating to sub-agent/workflow: "
                        f"<b>{transfer_target}</b></small>"
                    ))
            if event.content and event.content.parts:
                for part in event.content.parts:
                    if part.text:
                        append_message(
                            f"Agent ({author})",
                            part.text,
                            is_agent=True,
                        )
    except Exception as e:
        append_message("System", f"Error validation: {e}", is_agent=True)


user_text_input.on_submit(handle_submit)
send_button.on_click(handle_submit)
exit_button.on_click(lambda b: disable_interface())

clear_output()
ipy_display(css_widget)
ipy_display(interface_box)

with chat_output:
    ipy_display(widgets.HTML(
        f"<b style='color: #6c757d;'>Challenge 4 Workflow System initialized "
        f"(Project: {PROJECT_ID} | "
        f"root_agent -> [greeter_agent, weather_agent, "
        f"answer_team(search_agent -> critique_agent -> refine_agent)]). "
        f"Choose your model and type a query below:</b><br>"
    ))

HTML(value='\n<style>\n.widget-dropdown select {\n    background-color: #212529 !important;\n    color: #fffff…

In [ ]:
# ------------------------------------------------------------
# SECOND CHALLENGE 4 CODE CELL: Instruction 4
# Test agents & output events demonstrating sub-agents and
# SequentialAgent workflow (Search -> Critique -> Refine)
# ------------------------------------------------------------


def test_challenge4_workflow(prompt: str, test_session_id: str):
    """Run a test prompt and print all sub-agent/workflow events."""
    print("\n" + "=" * 75)
    print(f"CHALLENGE 4 TEST REQUEST: {prompt}")
    print("=" * 75)

    user_msg = types.Content(
        role="user",
        parts=[types.Part.from_text(text=prompt)],
    )
    events = runner_flash.run(
        user_id="challenge4_tester",
        session_id=test_session_id,
        new_message=user_msg,
    )

    for idx, event in enumerate(events, start=1):
        author = getattr(event, "author", "unknown")
        print(f"\n[Event #{idx}] Author: {author}")

        actions = getattr(event, "actions", None)
        transfer_target = getattr(actions, "transfer_to_agent", None)
        if transfer_target:
            print(f"  -> Action (Transfer to Sub-Agent): {transfer_target}")

        if event.content and event.content.parts:
            for part in event.content.parts:
                if getattr(part, "function_call", None):
                    fc = part.function_call
                    args_dict = dict(fc.args) if fc.args else {}
                    print(f"  -> Tool Call: {fc.name}({args_dict})")
                if getattr(part, "function_response", None):
                    fr = part.function_response
                    print(
                        f"  -> Tool Response from `{fr.name}`: {fr.response}"
                    )
                if getattr(part, "text", None):
                    print(f"  -> Text Output: {part.text.strip()}")


# Test A: Demonstrate Greeter Sub-Agent
test_challenge4_workflow(
    "Hello! What can you help me with today?",
    test_session_id=f"c4_greeter_{int(time.time())}",
)

# Test B: Demonstrate SequentialAgent Answer Team (Search -> Critique -> Refine)
test_challenge4_workflow(
    "Is Oak Hill a city in Northern Virginia or an unincorporated area, "
    "and how is it governed?",
    test_session_id=f"c4_workflow_{int(time.time())}",
)


CHALLENGE 4 TEST REQUEST: Hello! What can you help me with today?

[Event #1] Author: root_agent
  -> Tool Call: transfer_to_agent({'agent_name': 'greeter_agent'})

[Event #2] Author: root_agent
  -> Action (Transfer to Sub-Agent): greeter_agent
  -> Tool Response from `transfer_to_agent`: {'result': None}

[Event #3] Author: greeter_agent
  -> Text Output: Hello! I'm your friendly Greeter Agent. I can help you by providing US weather forecasts, or by answering complex questions using our 3-step Search, Critique, and Refine workflow team. How can I assist you today?

CHALLENGE 4 TEST REQUEST: Is Oak Hill a city in Northern Virginia or an unincorporated area, and how is it governed?

[Event #1] Author: root_agent
  -> Tool Call: transfer_to_agent({'agent_name': 'answer_team'})

[Event #2] Author: root_agent
  -> Action (Transfer to Sub-Agent): answer_team
  -> Tool Response from `transfer_to_agent`: {'result': None}

[Event #3] Author: search_agent
  -> Tool Call: google_search_worker(